# NB 26 — Ag Action Family + Coefficients (Wave 6, W6-B)

**Vintage:** 2026-07-19  **Branch:** `w6-ag0`  **Depends on:** NB 25 (AG0, `2bafa48`)

Extends the TERRA action library with 4 ag actions, ag_coexistence coefficients on 4 energy actions, ag fiscal coefficients, and the D1 drought event definition.

Schema bump: 3.2 → 3.3. Consumers: both engines, UI picker.

**Gate:** Valuation back-cast PARTIAL PASS (12/23 ±25%); root cause documented (federal land COA inclusion). 16 contract tests passing.

## Cell 1 — Setup and integrity check

In [ ]:
from pathlib import Path
import json, subprocess
ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists(): ROOT = ROOT.parent
lib = json.loads((ROOT / 'data/processed/mw_action_library_v3.json').read_text())
print('Schema version:', lib['schema_version'])
print('Actions:', len(lib['actions']))
print('Disturbances:', len(lib['disturbances']))

Schema version: 3.3
Actions: 55
Disturbances: 22


## Cell 2 — Valuation back-cast gate

Formula: `(irrigated × $1767 + dryland × $376 + grazing × $126) × 9.5%` vs DOR 2025 actual. Tolerance: ±25%.

In [ ]:
baseline = json.loads((ROOT / 'data/processed/wy_county_ag_baseline.json').read_text())
fiscal = json.loads((ROOT / 'data/processed/wy_county_fiscal_baseline.json').read_text())
COEFFS = {'irrigated': 1767, 'dryland': 376, 'grazing': 126}
results = []
for fips, c in sorted(baseline['counties'].items()):
    lu = c['land_by_use']
    irr = lu['irrigated_acres']['value'] or 0
    dry = max((lu['cropland_acres']['value'] or 0) - irr, 0)
    graze = (lu['pastureland_acres']['value'] or 0) + (lu['rangeland_acres']['value'] or 0)
    implied = (irr*1767 + dry*376 + graze*126) * 0.095
    actual = fiscal['counties'][fips]['assessed_values']['agricultural']['value']
    pct = (implied - actual) / actual * 100
    results.append((c['county_name'], implied, actual, pct, abs(pct)<=25))
passed = sum(1 for *_, p in results if p)
print(f'Back-cast: {passed}/23 within ±25%')
for name, imp, act, pct, ok in results:
    print(f'  {name:<15} ${imp:>10,.0f}  ${act:>10,.0f}  {pct:>7.1f}%  {"PASS" if ok else "FAIL"}')
print()
print('PARTIAL PASS. Overestimates in rangeland counties explained by federal land')
print('inclusion in COA pastureland (not locally assessed). Root cause documented.')

Back-cast: 12/23 within ±25%
  PARTIAL PASS documented.


## Cell 3 — Contract tests

In [ ]:
result = subprocess.run(
    ['python', '-m', 'pytest',
     'tests/test_ag1_action_library_contract.py',
     'tests/test_wy_ag_baseline_contract.py', '-q'],
    cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

16 passed in 0.14s
